# Preprocessing Technique: Text Cleaning & Normalization
### IT2011 — Progress Review I: Data Preprocessing and EDA
**Presented by:** Member 2 — *[D.S. Wijewardana, IT25101763]*
**Assigned dataset:** Rotten Tomatoes Movie Review Dataset (Cornell)

This notebook covers my individually-owned preprocessing technique for our group's project,
as required for Progress Review I: technique explanation, justification, implementation, and
an interpreted EDA visualization.


## Shared Setup

This cell is identical across every member's notebook so each person's notebook can run
independently. It loads the assigned dataset and converts it to a pandas DataFrame.


In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_style("whitegrid")

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)
train_df.head()


## 1. Technique Explanation

**Text cleaning / normalization** is the NLP equivalent of standardizing raw values in a
tabular dataset. It converts inconsistent, "messy" text into a consistent form: lowercasing,
removing punctuation noise, and collapsing extra whitespace — while deliberately preserving
tokens that carry sentiment meaning (like "not").

## 2. Justification for This Dataset

Without normalization, a model treats "Great!", "great", and "GREAT" as three completely
unrelated tokens, fragmenting the vocabulary and hiding the fact that they carry the same
sentiment. Standardizing casing and punctuation lets the model recognize these as the same
signal, which is especially important given our relatively small (10,662-row) dataset — every
duplicate-meaning token we can consolidate helps the model learn more from less data.


## 3. Implementation

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)   # strip punctuation, keep apostrophes
    text = re.sub(r"\s+", " ", text).strip()     # collapse extra whitespace
    return text
    # NOTE: negation words like "not" / "n't" are deliberately kept — they carry sentiment.

train_df["clean_text"] = train_df["text"].apply(clean_text)

# Show a few before/after examples
train_df[["text", "clean_text"]].sample(5, random_state=1)


## 4. EDA Visualization & Interpretation

To show the effect of normalization quantitatively, we compare the **vocabulary size**
(number of unique words) before and after cleaning.


In [ ]:
def vocab_size(series):
    words = set()
    for t in series:
        words.update(t.split())
    return len(words)

vocab_before = vocab_size(train_df["text"].str.lower())  # lowercase only, for a fairer comparison
vocab_after = vocab_size(train_df["clean_text"])

plt.figure(figsize=(5, 4))
sns.barplot(x=["Before cleaning\n(lowercased only)", "After cleaning\n(punctuation removed)"],
            y=[vocab_before, vocab_after], palette=["#C1272D", "#2E9E6D"])
plt.title("Vocabulary Size Before vs. After Text Cleaning")
plt.ylabel("Unique word count")
plt.show()

print(f"Vocabulary size before: {vocab_before}")
print(f"Vocabulary size after:  {vocab_after}")
print(f"Reduction: {vocab_before - vocab_after} unique tokens merged")


**Interpretation:** [Fill in after running — expected result: vocabulary size decreases
after cleaning, because punctuation-attached word variants (e.g. "great," / "great." / "great!")
collapse into a single token "great". A smaller, denser vocabulary means each remaining word
is seen more often during training, which generally helps a TF-IDF-based model learn more
reliable word-sentiment associations.]
